In [31]:
import math, pandas as pd
from math import gcd
from collections import Counter
from scipy.stats import chi2

def primeFactors(n):
    factors = set()
    i=2
    while i * i <= n:
        if n % i == 0:
            while n % i == 0:
                n //= i
        i += 1
    if n > 1:
        factors.add(n)
    return factors

#Test Hull-Dobbell (Vérifie si une séquence pseudo aléatoire est de période maximum)
def isFullPeriod(a, c, m):
    rule1 = gcd(c,m) == 1

    primeM = primeFactors(m)
    rule2 = all((a-1)%p == 0 for p in primeM)

    rule3 = (m % 4 !=0) or ((a-1) % 4 == 0)
    return rule1 and rule2 and rule3

#Formule congruentiel linéaire mixte (Générer la suite pseudo aléatoire)
def xnCompute(a, c , m, x0):
    xn = [x0]
    x1 = ((a * x0) + c) % m
    xn.append(x1)
    for i in range(m-1):
        xn.append(((a * xn[i]) + c) % m)
    return xn

#Test des fréquence
def unCompute(a,c,m,x0):
    xn = xnCompute(a,c,m,x0)
    un = []
    for i in range(len(xn)):
        un.append(xn[i] / m)
    return un

#Fréquence cumulée
def ynCompute(a,c,m,x0):
    un = unCompute(a,c,m,x0)
    yn =[]
    for i in range(len(un)):
        yn.append(int(un[i]*10))
    return yn

#test de saut (Savoir l'espace entre chaque nombre demandé dans la suite)
def jumpTest(a,c,m,x0, studiedNb):
    yn = ynCompute(a,c,m,x0)
    jump = []
    try:
        iPosition = yn.index(studiedNb)
    except ValueError:
        return jump 
    print(iPosition)

    for i in range(iPosition + 1, len(yn)):
        if(yn[i] == studiedNb):
            jump.append(i - iPosition - 1)
            iPosition = i
    return jump

#Test de course (Comparé les nombre 2 a 2 si le premier est supérieur au 2ème = 1 et si 1er supérieur = 2)
def courseTest(a, c, m, x0, size):
    xn = xnCompute(a, c, m, x0)
    course = []
    for i in range(0, size*2, 2):
        if xn[i] > xn[i + 1]:
            course.append(2)
        else:
            course.append(1)
    return course

#Permet de séparer le jeu de données and x groupe d'une taille y
def separating(a, c, m, x0, size, nbGroup):
    yn = ynCompute(a, c, m, x0)
    separated = []
    
    for i in range(0, size*nbGroup, size):
        temp = []
        for i in range(size):
            temp.append(yn[i])
        separated.append(temp)
    return separated

#compte le nombre de chaque combinaison possible dans un test de poker
def pokerCount(a,c,m,x0):
    suite = xnCompute(a,c,m,x0)
    poker = []
    #Dans l'ordre(Poker, Carré, Full, Brelan, Deux pair, une pair, rien)
    pokerCounter = [0,0,0,0,0,0,0]
    for i in range(0,len(suite), 5):
        if i + 5 <= len(suite):
            group= []
            for j in range(5):
                group.append(suite[i+j])
            poker.append(pokerCheck(group))
    for i in range(len(poker)):
        if poker[i] == "Poker":
            pokerCounter[0]+=1
        elif poker[i] == "Carré":
            pokerCounter[1]+=1
        elif poker[i] == "Full":
            pokerCounter[2]+=1
        elif poker[i] == "Brelan":
            pokerCounter[3]+=1
        elif poker[i] == "Deux Pair":
            pokerCounter[4]+=1
        elif poker[i] == "Une Pair":
            pokerCounter[5]+=1
        else:
            pokerCounter[6]+=1
    return pokerCounter

#Poker (dans groupe de 5 vérifie si il y a soit une pair, soit deux pair, soit un brelan(3 les memes) soit un carré(4 les memes) soit un full (une pair + un brelan) soit un poker(5 les memes) soit rien)
def pokerCheck(group):
    count = Counter(group)
    value = count.values()
    if 5 in value:
        return "Poker"
    elif 4 in value:
        return "Carré"
    elif 3 in value and 2 in value:
        return "Full"
    elif 3 in value:
        return "Brelan"
    elif list(value).count(2) ==2:
        return "Deux Pair"
    elif 2 in value:
        return "Une Pair"
    else:
        return "Rien"

#Compte combien de 0 et de 1 sont trouvé dans le test de course
def counting(a, c, m, x0):
    course = courseTest(a, c, m, x0)
    number = []
    one = 0
    two = 0
    for i in range(len(course)):
        if (course[i]) == 1:
            one+=1
        else:
            two+=1
    number.append(one)
    number.append(two)
    return number

#Test du carré-unité (Prendre nombre 4 a 4 pour en faire un graphique)
def carreUnit(a,c,m,x0, size):
    yn = ynCompute(a,c,m,x0)
    carreUnit = []
    for i in range(0, size*4, 4):
        carreUnit.append((yn[i+2] - yn[i])**2 + (yn[i+3] - yn[i+1])**2)
    return carreUnit

#Permet de faire la loi de poisson
def poisson(number):
    un = []
    unCumulated = []
    test = 0
    size = 0
    while(test < 1):
        temp1 = round((math.e** (-number)) * (number**size) / math.factorial(size), 3)
        test += temp1
        un.append(temp1)
        unCumulated.append(round(test, 3))
        size +=1
    return un, unCumulated

#Pas pertinent pour le projet je crois
def kCompute(x1, modulo, a, c):
    k=0
    while((x1 - c + k * modulo) % a != 0):
        k+=1
    return k

def grouping(xi,ri,pi,npi,x):
    i = 0
    while i < len(npi) - 1:
        if npi[i] < 5:
            xi[i] += " + " + xi[i+1]
            ri[i] += ri[i+1]
            pi[i] += pi[i+1]
            npi[i] += npi[i+1]
            x[i] += x[i+1]
            del xi[i+1], ri[i+1], pi[i+1], npi[i+1], x[i+1]
            if i > 0:
                i-=1
        else:
            i+=1
    if len(npi) == 1 and npi[0] <5:
        return False
    return xi, ri, pi, npi, x

In [ ]:
def generation(a, c, m, x0):

    full_period = isFullPeriod(a, c, m) 
    suite = xnCompute(a, c, m, x0)

    return(full_period,suite)

def poker(a,c,m,x0):
    print("Etape 1 :")
    print("H0 : la distribution des combinaisons (paire, double paire, brelan, etc.) correspond aux probabilités théoriques.")
    print("H1 : la distribution diffère de celle attendue")

    print("\nEtape2 :")
    alpha = 0.05
    print(alpha)

    print("\nEtape 3 :")
    
    xi = ["Poker", "Carré", "Full", "Brelan", "Deux Pair", "Une Pair", "Rien"]
    ri = pokerCount(a,c,m,x0)
    pi = [1/(10**4), 450/(10**5), 900/(10**5), 7200/(10**5), 10800/(10**5), 50400/(10**5), 0.3024]
    npi = [sum(ri)*p for p in pi]
    i = [(ri[j] - npi[j])**2 / npi[j] for j in range(len(ri))]

    dp_tab = pd.DataFrame({'xi': xi, 'ri': ri, 'pi' : pi, "npi" : npi, '(ri-npi)²/npi': i})
    print(dp_tab.to_string(index=False))

    print("\nEtape 4 :")
    grouped = grouping(xi,ri,pi,npi,i)
    if(grouped):
        dpg_tab = pd.DataFrame({'xi': xi, 'ri': ri, 'pi' : pi, "npi" : npi, '(ri-npi)²/npi': i})
        print(dpg_tab.to_string(index=False))
        nb_modalite = dpg_tab['ri'].sum()
        degre_liberte = nb_modalite - 1
        x2_obs_total = dpg_tab['(ri-npi)²/npi'].sum()
        print("\nX2 = %f" % x2_obs_total)
    else:
        print("Toujours inférieur a 5 aprés regroupement de toutes les catégories")
        nb_modalite = dp_tab['ri'].sum()
        degre_liberte = nb_modalite - 1
        x2_obs_total = dp_tab['(ri-npi)²/npi'].sum()
        print("\nX2 = %f" % x2_obs_total)
    
    print("\nEtape 5 :")

    valeur_critique = chi2.ppf(1 - alpha, degre_liberte)
    print("\nv = %f" % valeur_critique)

    print("\nEtape 6 :")
    
    if x2_obs_total <= valeur_critique:
        print("H0 est acceptée : la distribution des combinaisons (paire, double paire, brelan, etc.) correspond aux probabilités théoriques.")
    else:
        print("H1 est acceptée : la distribution diffère de celle attendue")
        
def partie1(a,c,m,x0):
    full_period, suite = generation(a, c, m, x0)

    if not full_period:
        print("Le 3 hypothèses du théorème de Hull-Dobell ne sont pas respectées")
    else:
        print("Les 3 hypothèses du théorème de Hull-Dobell sont respectées")
        print("\nTest de fréquence en six étapes :\n")
        frequence(a,c,m,x0)
        print("\nTest de poker en six étapes :\n")
        poker(a,c,m,x0)

Etape 1 :
H0 : la distribution des combinaisons (paire, double paire, brelan, etc.) correspond aux probabilités théoriques.
H1 : la distribution diffère de celle attendue

Etape2 :
0.05

Etape 3 :
       xi  ri     pi    npi  (ri-npi)²/npi
    Poker   0 0.0001 0.0007       0.000700
    Carré   0 0.0045 0.0315       0.031500
     Full   0 0.0090 0.0630       0.063000
   Brelan   0 0.0720 0.5040       0.504000
Deux Pair   7 0.1080 0.7560      51.570815
 Une Pair   0 0.5040 3.5280       3.528000
     Rien   0 0.3024 2.1168       2.116800

Etape 4 :
                                                         xi  ri  pi  npi  (ri-npi)²/npi
Poker + Carré + Full + Brelan + Deux Pair + Une Pair + Rien   7 1.0  7.0      57.814815

X2 = 57.814815

Etape 5 :

v = 12.591587
6

Etape 6 :
H1 est acceptée : la distribution diffère de celle attendue
